## Refactoring Checklist

### Issues Found
- [x] Generic error handling in the podcast pipeline
- [x] TTS failures were not clearly logged
- [x] PDF loading errors lacked contextual debugging information
- [x] Some functions handled multiple responsibilities
- [x] Old project setup code depended on a missing `src` directory
- [x] Unused imports and dead code existed in notebooks

### Priority
1. Improve error handling and debugging clarity
2. Reduce silent failures
3. Improve modularity and maintainability
4. Remove redundant/dead code

In [1]:
"""Main notebook for the podcast studio app."""

from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

DEFAULT_TTS_MODELS = ["tts-1", "tts-1-hd"]

def log_error(function_name: str, error: Exception, context: str = ""):
    error_message = (
        f"\nERROR in {function_name}\n"
        f"Error Type: {type(error).__name__}\n"
        f"Error Message: {error}\n"
        f"Context: {context}\n"
    )

    print(error_message)

    return error_message


def _get_pdf_reader():
    try:
        from PyPDF2 import PdfReader
        return PdfReader
    except ModuleNotFoundError:
        try:
            from pypdf import PdfReader
            return PdfReader
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Missing PDF reader dependency. Install it with:\n"
                "python -m pip install -r requirements.txt\n"
                "or install PyPDF2 directly with:\n"
                "python -m pip install PyPDF2"
            ) from exc


def _resolve_pdf_path(file_path: str) -> Path:
    pdf_path = Path(file_path)

    if pdf_path.exists():
        return pdf_path

    if not pdf_path.is_absolute():
        for base in [Path.cwd(), *Path.cwd().parents]:
            candidate = base / pdf_path
            if candidate.exists():
                return candidate

    raise FileNotFoundError(f"PDF not found: {file_path}")


class DataProcessor:
    def load_pdf(self, file_path: str) -> str:
        try:
            pdf_path = _resolve_pdf_path(file_path)
            text_output = ""
            PdfReader = _get_pdf_reader()

            with open(pdf_path, "rb") as file:
                reader = PdfReader(file)

                for page in reader.pages:
                    page_text = page.extract_text() or ""
                    text_output += page_text + "\n"

            return text_output

        except Exception as e:
            log_error(
                function_name="DataProcessor.load_pdf",
                error=e,
                context=f"Failed to load PDF: {file_path}"
            )
            raise

class LLMProcessor:
    def __init__(self):
        self.client = OpenAI()

    def summarize_text(self, text: str) -> str:
        prompt = f"""You are an educational podcast script writer.

You will receive structured notes extracted from a full class transcript.
Create a short recap podcast script for students who missed the lesson.

The recap must:
- reflect the whole lesson, not only the beginning
- prioritize the actual teaching topic
- include operational context only when it matters
- mention API key / quota issues briefly if they affected the class
- explain the main lesson concepts clearly
- sound natural when read aloud
- be around 3 to 5 minutes when spoken

Use this structure:

Title:
A short title.

Intro:
Briefly explain what happened in class and what the lesson focused on.

Main Lesson Recap:
Explain the main concepts in a clear learning order.

Operational Notes:
Briefly mention relevant setup/API/project issues if important.

Practical Takeaways:
List what students should remember or apply.

Outro:
End with a short closing sentence.
Your task is to extract useful information for an educational lesson recap.

Separate the content into these categories:

1. Lesson Content:
- Concepts taught
- Definitions
- Examples
- Tools, techniques, or workflows explained
- Practical advice students should remember

2. Operational Context:
- Important class logistics
- API key or setup issues
- Project constraints or blockers
- Anything that affected the lesson flow

3. Ignore:
- Small talk
- Repeated phrases
- Frozen screen comments
- Unrelated personal conversation
- Long troubleshooting details unless they affected the learning topic

Rules:
- Do not invent content.
- Keep important API or setup issues if they affected the class.
- Prioritize actual teaching content over administrative content.
- Be concise but specific.

        {text}
        """

        response = self.client.responses.create(
            model="gpt-4o-mini",
            input=prompt,
        )

        return response.output_text


class TTSGenerator:
    def __init__(self):
        self.client = OpenAI()

    def text_to_audio(self, text: str, output_path="output.mp3", model: str | None = None):
        speech_file_path = Path(output_path)
        candidate_models = []
        if model:
            candidate_models.append(model)
        candidate_models.extend(m for m in DEFAULT_TTS_MODELS if m not in candidate_models)

        last_error = None
        for candidate_model in candidate_models:
            try:
                with self.client.audio.speech.with_streaming_response.create(
                    model=candidate_model,
                    voice="alloy",
                    input=text,
                ) as response:
                    response.stream_to_file(speech_file_path)
                return str(speech_file_path)
            except Exception as exc:
                last_error = exc
                log_error(
                    function_name="TTSGenerator.text_to_audio",
                    error=exc,
                    context=f"TTS model failed: {candidate_model}. Trying next model."
                )
        
                continue

                raise RuntimeError(
                    "Unable to generate speech with any supported TTS model. "
                    "Tried: " + ", ".join(candidate_models)
                ) from last_error


data_processor = DataProcessor()
llm = LLMProcessor()
tts = TTSGenerator()


def _resolve_uploaded_file(pdf_file):
    if pdf_file is None:
        raise ValueError("No PDF file was uploaded.")

    if isinstance(pdf_file, (str, Path)):
        return str(pdf_file)

    if hasattr(pdf_file, "path") and pdf_file.path:
        return str(pdf_file.path)

    if hasattr(pdf_file, "name") and pdf_file.name:
        return str(pdf_file.name)

    raise TypeError(f"Unsupported uploaded file value: {type(pdf_file)!r}")


def generate_podcast(pdf_file, tts_model):
    try:
        print("Step 1: Resolving PDF path...")
        pdf_path = _resolve_uploaded_file(pdf_file)
        print(f"   -> PDF path resolved: {pdf_path}")

        print("Step 2: Extracting text from PDF...")
        text = data_processor.load_pdf(pdf_path)
        print(f"   -> Text extracted successfully ({len(text)} characters)")

        print("Step 3: Generating podcast script with LLM...")
        script = llm.summarize_text(text)
        print("   -> Script generated successfully")

        print("Step 4: Generating audio file...")
        audio_path = tts.text_to_audio(script, model=tts_model)
        print(f"   -> Audio generated: {audio_path}")

        print("Pipeline completed successfully!")
        return script, audio_path

    except Exception as e:
        error_message = log_error(
        function_name="generate_podcast",
        error=e,
        context="Podcast generation pipeline failed."
    )
    return error_message, None

with gr.Blocks() as demo:
    gr.Markdown("# Podcast Studio MVP")
    gr.Markdown("Upload a PDF -> Get a podcast script + audio")

    pdf_input = gr.File(label="Upload PDF")
    tts_model_input = gr.Dropdown(
        choices=DEFAULT_TTS_MODELS,
        value=DEFAULT_TTS_MODELS[0],
        label="TTS Model",
        info="Pick a speech model to try first.",
    )
    script_output = gr.Textbox(label="Generated Podcast Script")
    audio_output = gr.Audio(label="Generated Audio", type="filepath")

    generate_btn = gr.Button("Generate Podcast")

    generate_btn.click(
        fn=generate_podcast,
        inputs=[pdf_input, tts_model_input],
        outputs=[script_output, audio_output],
    )
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Refactoring and Error Handling Improvements

This project was refactored to improve readability, maintainability, modularity, and debugging clarity.

### Changes Made

- Added a reusable `log_error()` helper function to centralize error reporting.
- Improved error handling in `generate_podcast()` so pipeline failures clearly show:
  - where the error occurred,
  - the error type,
  - the error message,
  - and additional context.
- Refactored `DataProcessor.load_pdf()` to use explicit `try/except` handling and re-raise errors after logging.
- Improved `TTSGenerator.text_to_audio()` by logging failed TTS model attempts instead of failing silently.
- Preserved the existing Gradio UI and overall application behaviour.

### Why These Changes Matter

These changes make the code:

- **More modular**: responsibilities are separated into focused functions and classes.
- **More maintainable**: reusable helper functions reduce duplicated code.
- **Easier to debug**: errors now clearly show WHAT failed, WHERE it failed, and WHY.
- **More robust**: failures are handled explicitly instead of silently ignored.

This follows the refactoring principles covered in the lesson:
- modular design,
- helper functions,
- separation of concerns,
- and “fail loud, not silently” error handling.

## Testing

The refactored application was tested after each major change.

### Tests Performed
- PDF upload and extraction
- LLM podcast script generation
- TTS audio generation
- Full Gradio pipeline execution

### Result
The application behaviour remained unchanged while improving error handling, modularity, and maintainability.